[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/aipi590-challenge-3/blob/main/notebooks/challenge3.ipynb)

# AIPI 590 — Challenge 3: RL in the Physical World

**Task:** Train a manipulation policy in MuJoCo simulation (FetchReach-v3) using SAC + Hindsight Experience Replay, then analyze the sim-to-real gap against a real 6-DOF arm platform.

**Setup:** Add a Colab secret named `GITHUB_TOKEN_AIPI590_CHALLENGE_3` (key icon → left sidebar) with repo write access scoped to `jonasneves`.

## 0. Install

Run this cell first. If Colab prompts a runtime restart, restart and continue from the next cell — do not re-run this one.

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', '-r', '/content/aipi590-challenge-3/requirements.txt'], check=False)
# If requirements.txt not yet available, install directly:
subprocess.run([
    'pip', 'install', '-q',
    'mujoco>=3.0.0',
    'gymnasium-robotics>=1.2.0',
    'stable-baselines3[extra]>=2.3.0',
    'moviepy>=1.0.3',
    'imageio[ffmpeg]',
], check=True)

## 1. Setup

In [ ]:
BRANCH = 'main'  # @param {type:"string"}

import os, subprocess, sys
from pathlib import Path
import importlib.util

REPO = Path('/content/aipi590-challenge-3')

if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(
        ['git', 'clone', '--depth=1', '--branch', BRANCH,
         'https://github.com/jonasneves/aipi590-challenge-3.git', str(REPO)],
        check=True, env=env
    )

os.chdir(REPO)

spec = importlib.util.spec_from_file_location('colab_utils', REPO / 'scripts' / 'colab_utils.py')
colab_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_utils)
publish_artifacts = colab_utils.publish_artifacts

print(f'Repo ready at {REPO}')

In [ ]:
import gymnasium as gym
import gymnasium_robotics
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from pathlib import Path
import base64
from IPython.display import HTML, display

from stable_baselines3 import SAC
from stable_baselines3.her.her_replay_buffer import HerReplayBuffer
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor

gym.register_envs(gymnasium_robotics)

RESULTS = REPO / 'results'
MODELS_DIR = RESULTS / 'models'
VIDEOS_DIR = RESULTS / 'videos'
PLOTS_DIR  = RESULTS / 'plots'
for d in [MODELS_DIR, VIDEOS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Imports OK')

## 2. Simulation Environment

**FetchReach-v3** places a 7-DOF Fetch robot arm in a MuJoCo scene and asks it to move the end-effector to a randomly sampled 3D target position. The reward is sparse: `0` on success (distance < 5 cm), `-1` otherwise. This makes vanilla RL nearly impossible without goal-conditioned hindsight — which is why HER is the right pairing.

In [ ]:
env = gym.make('FetchReach-v3')
obs, _ = env.reset()

print('Observation keys:', list(obs.keys()))
print('  observation shape:', obs['observation'].shape)
print('  achieved_goal shape:', obs['achieved_goal'].shape)
print('  desired_goal shape:', obs['desired_goal'].shape)
print('Action space:', env.action_space)
env.close()

## 3. Training: SAC + HER

**Why SAC?** Off-policy, sample-efficient, handles continuous action spaces well. Entropy regularization prevents premature convergence — useful when the reward signal is almost always `-1` early in training.

**Why HER?** Hindsight Experience Replay relabels failed trajectories as successes toward the goal that was actually reached. It converts sparse-reward failures into dense learning signal without reward shaping. `goal_selection_strategy='future'` selects relabeling goals from states later in the same episode — empirically the strongest variant (Andrychowicz et al., 2017).

In [ ]:
TOTAL_TIMESTEPS = 200_000  # ~25 min on Colab T4

train_env = Monitor(gym.make('FetchReach-v3'))
eval_env  = Monitor(gym.make('FetchReach-v3'))

model = SAC(
    'MultiInputPolicy',
    train_env,
    replay_buffer_class=HerReplayBuffer,
    replay_buffer_kwargs=dict(
        n_sampled_goal=4,
        goal_selection_strategy='future',
    ),
    verbose=1,
    buffer_size=int(1e6),
    learning_rate=1e-3,
    gamma=0.95,
    batch_size=256,
    policy_kwargs=dict(net_arch=[256, 256, 256]),
    tensorboard_log=str(RESULTS / 'tb'),
)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=str(MODELS_DIR),
    log_path=str(RESULTS / 'eval_logs'),
    eval_freq=10_000,
    n_eval_episodes=20,
    verbose=1,
)

model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_callback)
model.save(str(MODELS_DIR / 'sac_her_fetchreach_final'))
print('Training complete.')

In [ ]:
eval_log = np.load(str(RESULTS / 'eval_logs' / 'evaluations.npz'))

timesteps   = eval_log['timesteps']
mean_reward = eval_log['results'].mean(axis=1)
success_rate = (eval_log['results'] == 0).mean(axis=1)  # reward=0 means success

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(timesteps, mean_reward)
ax1.set_xlabel('Timesteps')
ax1.set_ylabel('Mean episode reward')
ax1.set_title('FetchReach-v3 — SAC+HER')
ax1.grid(alpha=0.3)

ax2.plot(timesteps, success_rate)
ax2.set_xlabel('Timesteps')
ax2.set_ylabel('Success rate')
ax2.set_ylim(0, 1)
ax2.set_title('FetchReach-v3 — Success Rate')
ax2.grid(alpha=0.3)

fig.tight_layout()
plot_path = PLOTS_DIR / 'training_curves.png'
fig.savefig(str(plot_path), dpi=150)
plt.show()
print(f'Saved to {plot_path}')

## 4. Evaluation

In [ ]:
model = SAC.load(str(MODELS_DIR / 'best_model'), env=gym.make('FetchReach-v3'))

record_env = gym.make('FetchReach-v3', render_mode='rgb_array')
record_env = gym.wrappers.RecordVideo(
    record_env,
    str(VIDEOS_DIR),
    episode_trigger=lambda ep: True,
    name_prefix='fetchreach',
)

n_episodes = 5
successes = 0
for ep in range(n_episodes):
    obs, _ = record_env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = record_env.step(action)
        done = terminated or truncated
    if info.get('is_success', False):
        successes += 1

record_env.close()
print(f'Success rate: {successes}/{n_episodes}')

videos = sorted(Path(VIDEOS_DIR).glob('*.mp4'))
if videos:
    video_bytes = videos[-1].read_bytes()
    b64 = base64.b64encode(video_bytes).decode()
    display(HTML(f'<video controls width="480"><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
else:
    print('No video found — check VIDEOS_DIR.')

## 5. Sim-to-Real Analysis

Training a policy in MuJoCo and deploying it on a physical arm exposes a structured set of gaps. The analysis below is grounded in the **reBot-DevArm** platform (6-DOF, Robostride/Damiao actuators, ROS 2 Humble), which represents a realistic target for deploying a policy trained in an environment like FetchReach-v3.

---

### 5.1 Actuator Fidelity

FetchReach-v3 uses **idealized velocity control**: the policy outputs a 4D delta in Cartesian space and MuJoCo solves inverse kinematics internally, applying exact joint velocities at every simulation step (~2 ms). Real actuators are substantially messier:

- **Gear backlash.** Robostride and Damiao motors use cycloidal or harmonic reducers with measurable dead-band. Commands below the dead-band threshold produce no motion; commands above it produce a step. The policy never encounters this in simulation.
- **Torque limits and thermal drift.** Real motors derate as they heat. A joint near its torque limit in simulation may saturate unpredictably on hardware after several minutes of operation.
- **Control loop latency.** The reBot-DevArm's ROS 2 control loop runs at ~100 Hz (~10 ms). MuJoCo steps are sub-millisecond. A policy trained at sim timestep resolution is implicitly assuming zero actuator lag — a gap that compounds across a 7-step reaching trajectory.

**Mitigation:** randomize actuator delay (`action_delay` in `gymnasium_robotics` or custom wrapper), add gear backlash noise to the action, and cap joint velocity commands to match real motor specs.

---

### 5.2 Contact and Collision Modeling

MuJoCo models contacts as convex-hull collisions with soft constraint forces. For a pure reaching task (no contact with the goal), this is largely irrelevant. It becomes the dominant failure mode the moment the task extends to **grasping** — the next natural step from FetchReach to FetchPickAndPlace. Real gripper finger compliance, object surface friction variability, and micro-slip during grasp are not captured by MuJoCo's rigid-body contact model.

**Mitigation:** For reaching, contact modeling error is minimal. For any extension to manipulation, system identification of friction parameters and gripper compliance is required before sim-trained policies transfer reliably.

---

### 5.3 Observation Noise

The FetchReach policy receives:
- **Ground-truth joint positions and velocities** — from the MuJoCo state vector, noiseless.
- **Ground-truth goal position** — a 3D coordinate with no uncertainty.

On the reBot-DevArm:
- Joint positions come from **magnetic encoders** (Damiao series: ~0.1° resolution) with electrical noise at high speeds. Velocity is numerically differentiated, amplifying noise.
- The goal position is typically estimated from a **depth camera** (e.g., RealSense D435), introducing 2–5 mm of spatial uncertainty and latency from the vision pipeline.

A policy trained on clean simulation observations will not gracefully degrade when these noise sources appear at deployment. The policy has learned to exploit the absence of noise rather than be robust to it.

**Mitigation:** add Gaussian noise to joint observations during training (`ObservationNoise` wrapper), and add positional noise to goal observations to simulate camera uncertainty.

---

### 5.4 Zero Calibration Drift

reBot-DevArm requires **explicit zero-point calibration** at each power cycle. Small per-joint calibration errors (even 1–2°) compound through the kinematic chain: at 650 mm reach, a 2° wrist error produces ~22 mm of end-effector position error — well above the 50 mm success threshold of FetchReach. The policy will consistently undershoot or overshoot.

This is not a sim2real gap in the conventional sense — it is an operational condition the simulation never models. No domain randomization covers it; it requires a real-world calibration procedure before policy deployment.

---

### 5.5 Domain Randomization Strategy

Given the gaps above, a minimal domain randomization scheme for this task:

| Parameter | Range | Rationale |
|---|---|---|
| Action delay | 0–20 ms | Matches ROS 2 control loop jitter |
| Joint obs noise | σ = 0.01 rad | Matches encoder resolution |
| Goal obs noise | σ = 3 mm | Matches depth camera uncertainty |
| Link mass | ±10% | Manufacturing tolerance |
| Joint friction | 0.5–2× nominal | Thermal and wear variation |

Training with this randomization set trades peak simulation performance for robustness — the policy will score slightly lower on the clean FetchReach benchmark but transfer more reliably to hardware.

---

### 5.6 Beyond Domain Randomization

Domain randomization is a coverage argument: if the real world falls within the randomized distribution, the policy generalizes. When the distribution is poorly calibrated — which is common for a new platform like reBot-DevArm before system identification is complete — two additional approaches are relevant:

- **Residual policy learning:** Deploy the sim-trained policy on hardware, collect real trajectories, train a small residual correction network on real data. The base policy handles the bulk of the task; the residual corrects systematic biases (e.g., persistent calibration offset).
- **Real-to-sim adaptation (system identification):** Use real hardware trajectories to estimate the true actuator dynamics and friction parameters, update the simulation accordingly, and retrain. The reBot-DevArm's Isaac Sim integration (planned Q2 2026) is the intended path for this on this platform.

## 6. Publish Artifacts

In [ ]:
artifacts = [
    'results/plots/training_curves.png',
]

# Include the best video if it was recorded
videos = sorted(Path(VIDEOS_DIR).glob('*.mp4'))
if videos:
    rel = videos[-1].relative_to(REPO)
    artifacts.append(str(rel))

publish_artifacts(
    artifacts,
    'add training curves and rollout video',
    branch=BRANCH,
)